# Colab: live PPO preflight
Upload the ZIP to MyDrive/cr_rl_live_ppo/. Place future complete sessions in its sessions/ subdirectory. This notebook verifies/replays only.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/cr_rl_live_ppo')
bundle = drive_root / 'crrl_live_ppo_colab.zip'
assert bundle.is_file(), f'Upload {bundle} first'


In [ ]:
import hashlib, json, shutil, zipfile
root = Path('/content/crrl_live_ppo')
with zipfile.ZipFile(bundle) as z:
    manifest = json.loads(z.read('BUNDLE_MANIFEST.json'))
    assert shutil.disk_usage('/content').free > sum(x['bytes'] for x in manifest) + 1024**3
    for row in manifest:
        target = (root / row['path']).resolve()
        assert target.is_relative_to(root.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        tmp = target.with_suffix(target.suffix + '.partial')
        h = hashlib.sha256()
        with z.open(row['path']) as src, tmp.open('wb') as dst:
            for block in iter(lambda: src.read(8 * 1024**2), b''):
                h.update(block)
                dst.write(block)
        assert tmp.stat().st_size == row['bytes'] and h.hexdigest() == row['sha256'], row['path']
        tmp.replace(target)
print('Bundle verified.')


In [ ]:
import os, subprocess, sys, torch
source = drive_root / 'sessions'
sessions = sorted(p for p in source.glob('*') if (p/'result.json').is_file())
if not sessions:
    print('No labeled recordings yet; upload complete session folders before running replay.')
else:
    assert torch.cuda.is_available(), 'Switch to a GPU runtime for FP16 replay.'
    local = [shutil.copytree(p, root/'sessions'/p.name, dirs_exist_ok=True) for p in sessions]
    env = dict(os.environ, PYTHONPATH=str(root/'src'))
    subprocess.run([sys.executable, 'scripts/check_nulls_ppo_batch.py', *(str(p) for p in local),
                    '--checkpoint', 'ppo_latest.pt'], cwd=root, env=env, check=True)
